# AgentSign -- Zero Trust Identity for AI Agents

**Try AgentSign in 30 seconds. No install. No setup. Free.**

This notebook starts the AgentSign Zero Trust Engine on a free Google Colab GPU and demos:
- Agent registration & identity pipeline
- Cryptographic passports (offline-verifiable)
- Signed execution chains (tamper-proof)
- MCP Trust Layer (ALLOW/DENY gate)
- Trust scoring (0-100 behavioral)
- Swarm management & revocation

**Links:** [Website](https://agentsign.dev) | [GitHub SDK](https://github.com/razashariff/agentsign-sdk) | [GitHub Server](https://github.com/razashariff/agentsign) | [npm](https://www.npmjs.com/package/agentsign)

MIT Licensed. Patent Pending (GB2604808.2).

---

## 1. Install & Start the AgentSign Engine

In [ ]:
# Install dependencies
!pip install -q fastapi uvicorn httpx aiosqlite cryptography pyngrok nest-asyncio python-multipart stripe

# Clone the AgentSign server
!git clone -q https://github.com/razashariff/agentsign.git /content/agentsign 2>/dev/null || echo 'Already cloned'

print('Dependencies installed.')

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import subprocess, time, os, httpx, json

# Start the AgentSign engine in the background
os.chdir('/content/agentsign')
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8888'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for server to be ready
for i in range(15):
    try:
        r = httpx.get('http://localhost:8888/api/engine/stats', timeout=2)
        if r.status_code == 200:
            print(f'AgentSign Engine running on port 8888')
            stats = r.json()
            print(f'  Version: Zero Trust Engine')
            print(f'  Agents: {stats.get("total_agents", 0)}')
            print(f'  Pipeline stages: INTAKE -> VETTING -> TESTING -> DEV_APPROVED -> PROD_APPROVED -> ACTIVE')
            break
    except:
        time.sleep(1)
else:
    print('Server failed to start. Check logs:')
    print(proc.stderr.read().decode()[:500])

## 2. Register an Agent (enters Identity Pipeline)

In [ ]:
BASE = 'http://localhost:8888'

# Register a new agent -- enters INTAKE stage
r = httpx.post(f'{BASE}/api/agents/onboard', json={
    'name': 'Procurement Bot',
    'description': 'Handles vendor payments and invoice processing',
    'category': 'finance',
    'code': 'def process_invoice(vendor, amount): return {"paid": True, "tx": "tx_" + vendor}',
    'permissions': ['execute', 'read_db', 'call_api']
})

agent = r.json()
AGENT_ID = agent['agent_id']

print(f'Agent registered!')
print(f'  ID:             {AGENT_ID}')
print(f'  Name:           {agent["name"]}')
print(f'  Pipeline Stage: {agent["pipeline_stage"]}')
print(f'  Trust Score:    {agent["trust_score"]}')
print(f'  Code Hash:      {agent.get("code_hash", "N/A")[:32]}...')

## 3. Advance Through Identity Pipeline

Every agent must pass through cryptographic vetting stages before reaching ACTIVE.
Each stage gate is recorded in the audit trail.

In [ ]:
# Advance through each pipeline stage
stages = ['VETTING', 'TESTING', 'DEV_APPROVED', 'PROD_APPROVED', 'ACTIVE']

for stage in stages:
    r = httpx.post(f'{BASE}/api/agents/{AGENT_ID}/advance')
    data = r.json()
    score = data.get('trust_score', '?')
    current = data.get('pipeline_stage', stage)
    print(f'  Stage: {current:<15} Trust Score: {score}')

print(f'\nAgent is now ACTIVE with trust score {score}/100')

## 4. Get Agent Passport (Offline-Verifiable)

The passport is a self-contained signed JSON. Any system can verify it **without contacting the server**.
The agent carries this everywhere -- it's cryptographic proof of identity.

In [ ]:
r = httpx.get(f'{BASE}/api/agents/{AGENT_ID}/passport')
passport = r.json()

print('AGENT PASSPORT')
print('=' * 60)
print(json.dumps(passport, indent=2))
print('=' * 60)
print(f'\nThis passport can be verified OFFLINE by any system.')
print(f'No server dependency. No network calls needed.')

## 5. Verify Passport Offline

In [ ]:
# Verify the passport (this would work even if the server was down)
r = httpx.post(f'{BASE}/api/verify/passport', json=passport)
verification = r.json()

print(f'Passport Verification: {verification["status"]}')
print(f'  Agent:    {verification.get("agent_name", "?")}')
print(f'  Revoked:  {verification.get("revoked", False)}')
print(f'  Tampered: {not verification.get("signature_valid", True)}')

# Now tamper with it and verify again
tampered = passport.copy()
tampered['trust_score'] = 100  # try to boost trust score
r2 = httpx.post(f'{BASE}/api/verify/passport', json=tampered)
print(f'\nTampered Passport: {r2.json()["status"]} (tamper detected!)')

## 6. Sign Executions (Tamper-Proof Chain)

Every input/output pair is cryptographically signed into a chain.
Creates a DAG of what the agent did -- tamper-proof audit trail.

In [ ]:
# Execute and sign -- agent processes an invoice
r = httpx.post(f'{BASE}/api/agents/{AGENT_ID}/execute', json={
    'input_data': {'action': 'process_invoice', 'vendor': 'Anthropic', 'amount': 35000},
    'output_data': {'status': 'paid', 'tx_id': 'tx_anth_001', 'timestamp': '2026-03-10T16:00:00Z'}
})
exec1 = r.json()

print('Signed Execution #1')
print(f'  Execution ID:   {exec1.get("execution_id", "?")}')
print(f'  Input Hash:     {exec1.get("input_hash", "?")[:32]}...')
print(f'  Output Hash:    {exec1.get("output_hash", "?")[:32]}...')
print(f'  Execution Hash: {exec1.get("execution_hash", "?")[:32]}...')
print(f'  Signature:      {exec1.get("signature", "?")[:32]}...')

# Second execution -- chained to the first
r2 = httpx.post(f'{BASE}/api/agents/{AGENT_ID}/execute', json={
    'input_data': {'action': 'process_invoice', 'vendor': 'OpenAI', 'amount': 12000},
    'output_data': {'status': 'paid', 'tx_id': 'tx_oai_002'}
})
exec2 = r2.json()
print(f'\nSigned Execution #2 (chained)')
print(f'  Execution Hash: {exec2.get("execution_hash", "?")[:32]}...')
print(f'\nBoth executions are now in the tamper-proof chain.')

## 7. MCP Trust Layer -- THE GATE

Before an agent can use ANY MCP tool, it presents its passport.
AgentSign checks identity + trust score + pipeline stage = ALLOW or DENY.

This is the killer feature.

In [ ]:
# First, register an MCP server (e.g. a database tool server)
r = httpx.post(f'{BASE}/api/mcp/servers/register', json={
    'name': 'Database MCP',
    'tools': ['query_users', 'insert_record', 'delete_record'],
    'trust_threshold': 60
})
mcp = r.json()
MCP_ID = mcp.get('server_id', mcp.get('id', 'unknown'))
print(f'MCP Server registered: {MCP_ID}')
print(f'  Tools: query_users, insert_record, delete_record')
print(f'  Trust Threshold: 60 (agent must score >= 60 to access)')

# Now the agent presents its passport to access the tool
print(f'\n--- Agent presents passport to MCP Trust Gate ---')
r = httpx.post(f'{BASE}/api/mcp/verify', json={
    'agent_id': AGENT_ID,
    'mcp_server_id': MCP_ID,
    'tool_name': 'query_users'
})
gate = r.json()
print(f'  Decision:     {gate.get("decision", "?")}')
print(f'  Trust Score:  {gate.get("trust_score", "?")}')
print(f'  Checks:       {gate.get("checks_passed", [])}')
print(f'\nThe agent was {gate.get("decision", "?").upper()}ED access to query_users.')

## 8. Revoke an Agent (Instant)

Revoke a compromised agent instantly. Trust score drops to 0. All MCP access denied.

In [ ]:
# Register a second agent to revoke
r = httpx.post(f'{BASE}/api/agents/onboard', json={
    'name': 'Rogue Bot',
    'description': 'This agent will be revoked',
    'category': 'test'
})
rogue = r.json()
ROGUE_ID = rogue['agent_id']

# Advance it to ACTIVE
for _ in range(5):
    httpx.post(f'{BASE}/api/agents/{ROGUE_ID}/advance')

# Now revoke it
r = httpx.post(f'{BASE}/api/agents/{ROGUE_ID}/revoke', json={'reason': 'Suspicious API calls detected'})
revoked = r.json()
print(f'Agent REVOKED: {ROGUE_ID}')
print(f'  Trust Score:    {revoked.get("trust_score", 0)}')
print(f'  Pipeline Stage: {revoked.get("pipeline_stage", "REVOKED")}')

# Try to access MCP -- should be DENIED
r = httpx.post(f'{BASE}/api/mcp/verify', json={
    'agent_id': ROGUE_ID,
    'mcp_server_id': MCP_ID,
    'tool_name': 'query_users'
})
print(f'\nRevoked agent tries MCP access: {r.json().get("decision", "DENY")}')
print(f'Reason: {r.json().get("reason", "Agent revoked")}')

## 9. Swarm Management

Group agents into swarms. Revoke an entire swarm with one call.

In [ ]:
# Create a swarm
r = httpx.post(f'{BASE}/api/swarms/create', json={
    'name': 'Finance Swarm',
    'description': 'All finance-related agents'
})
swarm = r.json()
SWARM_ID = swarm.get('swarm_id', swarm.get('id', 'unknown'))
print(f'Swarm created: {SWARM_ID} (Finance Swarm)')

# Add our agent to the swarm
r = httpx.post(f'{BASE}/api/swarms/{SWARM_ID}/add/{AGENT_ID}')
print(f'Added {AGENT_ID} to swarm')

# Create and add more agents
for name in ['Invoice Bot', 'Expense Bot']:
    r = httpx.post(f'{BASE}/api/agents/onboard', json={'name': name, 'category': 'finance'})
    aid = r.json()['agent_id']
    for _ in range(5):
        httpx.post(f'{BASE}/api/agents/{aid}/advance')
    httpx.post(f'{BASE}/api/swarms/{SWARM_ID}/add/{aid}')
    print(f'Added {name} ({aid}) to swarm')

print(f'\nSwarm has 3 agents. One command revokes them all.')

## 10. Local Signing with Python SDK (Zero Network)

The SDK signs locally. No server calls. No network dependency. Zero external dependencies.

In [ ]:
import sys
sys.path.insert(0, '/content/agentsign/sdk')
from agentsign import AgentSign

# Create a local agent (no server needed)
agent = AgentSign(
    name='Local Signer',
    description='Signs executions locally with zero network calls',
    code='def analyze(data): return {"risk": "low"}'
)

print(f'Local Agent: {agent.agent_id}')
print(f'Code Hash:   {agent.code_hash[:32]}...')
print(f'Certificate: {agent.certificate.cert_hash[:32]}...')

# Sign an execution locally
signed = agent.sign(
    input_data={'customer': 'Acme Corp', 'query': 'risk assessment'},
    output_data={'risk_level': 'low', 'score': 0.15, 'recommendation': 'approve'}
)

print(f'\nSigned Execution:')
print(f'  Exec Hash:  {signed.execution_hash[:32]}...')
print(f'  Signature:  {signed.signature[:32]}...')
print(f'  Verified:   {agent.verify(signed)}')

# Verify output integrity
print(f'  Output OK:  {agent.verify_output({"risk_level": "low", "score": 0.15, "recommendation": "approve"}, signed)}')

# Tamper with output
print(f'  Tampered:   {agent.verify_output({"risk_level": "high", "score": 0.95}, signed)}')

# Code attestation
print(f'  Code OK:    {agent.attest_code("def analyze(data): return {\"risk\": \"low\"}")}')  
print(f'  Modified:   {agent.attest_code("def analyze(data): return {\"risk\": \"HACKED\"}")}') 

## 11. Engine Stats

In [ ]:
r = httpx.get(f'{BASE}/api/engine/stats')
stats = r.json()

print('AGENTSIGN ENGINE STATS')
print('=' * 40)
for k, v in stats.items():
    print(f'  {k:<25} {v}')

print(f'\n--- Pipeline Summary ---')
r = httpx.get(f'{BASE}/api/pipeline/summary')
for stage, count in r.json().items():
    bar = '#' * count
    print(f'  {stage:<15} {count:>3} {bar}')

## 12. Make it Public (Optional -- ngrok tunnel)

Expose your AgentSign engine to the internet so remote agents can connect.

In [ ]:
# Uncomment and add your ngrok token to expose publicly
# Get a free token at https://dashboard.ngrok.com/signup

# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')
# tunnel = ngrok.connect(8888)
# print(f'AgentSign Engine is now PUBLIC at: {tunnel.public_url}')
# print(f'Any agent can connect with: AgentSign({{ serverUrl: "{tunnel.public_url}" }})')

print('To expose publicly, uncomment above and add your ngrok token.')
print('Get a free token at https://dashboard.ngrok.com/signup')

---

## What You Just Saw

| Feature | What It Does |
|---------|-------------|
| **Identity Pipeline** | INTAKE -> VETTING -> TESTING -> DEV_APPROVED -> PROD_APPROVED -> ACTIVE |
| **Agent Passport** | Self-contained signed JSON, verifiable offline |
| **Execution Chains** | Signed DAG of every input/output -- tamper-proof |
| **MCP Trust Layer** | ALLOW/DENY gate for tool access based on identity + trust |
| **Trust Scoring** | 0-100 behavioral score |
| **Swarm Management** | Group agents, revoke entire swarms instantly |
| **Local Signing** | Zero network, zero deps, HMAC-SHA256 |
| **Code Attestation** | Detect runtime code modifications |

### Next Steps

- **npm SDK**: `npm install agentsign` (Node.js, zero deps)
- **Self-host**: `docker run -d -p 8888:8888 ghcr.io/razashariff/agentsign:latest`
- **HSM support**: PKCS#11, AWS KMS, Azure Key Vault, GCP Cloud KMS, HashiCorp Vault
- **Website**: [agentsign.dev](https://agentsign.dev)
- **GitHub**: [razashariff/agentsign-sdk](https://github.com/razashariff/agentsign-sdk)

**Star the repo if you think AI agents need identity.**

MIT Licensed. Patent Pending (GB2604808.2). Built by [CyberSecAI](https://cybersecai.com).